In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

data_dir = r"D:\ImageNet"
img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=img_size, batch_size=batch_size
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=img_size, batch_size=batch_size, shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

base_model = ResNet50(weights="imagenet", include_top=False, pooling="avg")
base_model.trainable = False

def extract_features(ds):
    features, labels = [], []
    for images, y in ds:
        images = preprocess_input(images.numpy())
        features.append(base_model.predict(images, verbose=0))
        labels.append(y.numpy())
    return np.concatenate(features), np.concatenate(labels)

X_train, y_train = extract_features(train_ds)
X_test, y_test = extract_features(test_ds)

print("Feature-vector dimension:", X_train.shape[1])
np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)

classifier = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

classifier.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])
classifier.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1)

pred = np.argmax(classifier.predict(X_test), axis=1)

print(classification_report(y_test, pred, labels=np.arange(num_classes),
                            target_names=class_names, zero_division=0))

cm = confusion_matrix(y_test, pred, labels=np.arange(num_classes))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    xticks_rotation=90, cmap="Blues"
)
plt.tight_layout()
plt.show()

loss, accuracy = classifier.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", accuracy)

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

data_path = "D:/ImageNet/Images"
img_size = (224, 224)
num_classes = 10

images, labels = [], []
classes = sorted([d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))])

for c in classes:
    files = [f for f in os.listdir(os.path.join(data_path, c))
             if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))]
    for f in files[:100]:
        images.append(img_to_array(load_img(os.path.join(data_path, c, f), target_size=img_size)))
        labels.append(c)

unique_classes = sorted(set(labels))[:num_classes]
selected = [i for i, label in enumerate(labels) if label in unique_classes]
images = np.array(images)[selected]
labels = np.array(labels)[selected]

class_names = sorted(set(labels))
class_to_id = {c: i for i, c in enumerate(class_names)}
y = np.array([class_to_id[c] for c in labels])

x_train, x_test, y_train, y_test = train_test_split(
    images, y, test_size=0.2, random_state=42, stratify=y
)

base_model = ResNet50(weights="imagenet", include_top=False, pooling="avg")
base_model.trainable = False

x_train = preprocess_input(x_train)
x_test = preprocess_input(x_test)

features_train = base_model.predict(x_train, batch_size=32, verbose=1)
features_test = base_model.predict(x_test, batch_size=32, verbose=1)

print("Training feature shape:", features_train.shape)
print("Testing feature shape:", features_test.shape)
print("Feature vector dimension:", features_train.shape[1])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(features_train.shape[1],)),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(features_train, y_train, epochs=15, batch_size=32, validation_split=0.1, verbose=1)

probabilities = model.predict(features_test, verbose=0)
y_pred = np.argmax(probabilities, axis=1)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="macro", zero_division=0))
print("Recall:", recall_score(y_test, y_pred, average="macro", zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, average="macro", zero_division=0))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(xticks_rotation=90)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()
